# Describe maps — interactive viewer

Replay the precomputed TAM annotations written by `tamart.experiments.describe`.
No model inference is run here — maps, tokens and the processed image are read
straight from `data/datasets/wikiart_most_viewed/describe/<stem>/`.

Pick an image from the dropdown, then click below the painting to see activation
maps. Two modes:

- **Token mode** (default): click individual generated tokens to overlay their
  (mean) activation maps.
- **Span mode**: tick *Span mode* to use the classified expressions from
  `tamart.experiments.classify` (`classify/<stem>/classification.json`). The
  full caption is shown with the multi-token spans highlighted by category;
  click a single span to see the mean map over its tokens.

Tweak colormap / alpha / value range to inspect what each map contains.

In [1]:
import tamart  # must come before any transformers import (sets HF_HOME)
from tamart.tam import TAMExplainer

explainer = TAMExplainer()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Pick a precomputed annotation

The dropdown lists every subfolder under `describe/` that has an `answer.json`.

In [2]:
from pathlib import Path
import ipywidgets as widgets

DESCRIBE_DIR = Path("../data/datasets/wikiart_most_viewed/describe")
CLASSIFY_DIR = Path("../data/datasets/wikiart_most_viewed/classify")

folders = sorted(
    p for p in DESCRIBE_DIR.iterdir()
    if (p / "answer.json").exists()
)
assert folders, f"No precomputed annotations found in {DESCRIBE_DIR.resolve()}"

folder_dd = widgets.Dropdown(
    options=[(p.name, p) for p in folders],
    description="Image:",
    layout=widgets.Layout(width="60%"),
)
span_mode = widgets.Checkbox(
    value=False,
    description="Span mode (use classify/ expressions)",
    indent=False,
)
widgets.VBox([folder_dd, span_mode])

## Interactive viewer

Run the cell below after choosing an image above. Re-run it whenever you change
the dropdown selection to load a different annotation.

In [ ]:
classify_folder = CLASSIFY_DIR / folder_dd.value.name
use_spans = span_mode.value and (classify_folder / "classification.json").exists()
if span_mode.value and not use_spans:
    print(f"No classification.json under {classify_folder} — falling back to token mode.")

result = explainer.explain_interactive_precomputed(
    folder_dd.value,
    classify_folder=classify_folder if use_spans else None,
)